In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from pathlib import Path
import sys

sys.path.append('..')

from src.config import DATA_RAW

In [2]:
df = pd.read_csv(DATA_RAW / 'HR-Employee-Attrition.csv')
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


# Statical Testing

### Statistical Testing for Feature Significance

To formally assess the relationship between each feature and 'Attrition', we will perform statistical tests:

-   **Categorical Features**: Chi-squared test of independence to see if there's a significant association.
-   **Numerical Features**: Mann-Whitney U test to compare the distributions of the feature for employees who attrited vs. those who did not. This non-parametric test is chosen for its robustness, as it doesn't assume normal distribution of data.

We will use a significance level (alpha) of 0.05. A p-value less than 0.05 will indicate that the feature is statistically significant.

In [3]:
from scipy.stats import chi2_contingency, mannwhitneyu

results = []
alpha = 0.05

# Prepare the target variable (Attrition) for easier use in statistical tests
df_stat = df.copy()
# For numerical tests, we don't need to convert 'Attrition' to 0/1 yet, as Mann-Whitney U compares groups

for col in df_stat.columns:
    if col == 'Attrition':
        continue

    # Exclude identifier and constant columns that would not yield meaningful statistical results
    if col in ['EmployeeCount', 'StandardHours', 'EmployeeNumber', 'Over18']:
        continue

    # Check if the feature is categorical or numerical
    if df_stat[col].dtype == 'object' or df_stat[col].nunique() < 20: # Heuristic: if less than 20 unique values, treat as categorical
        # Categorical Feature: Chi-squared test
        contingency_table = pd.crosstab(df_stat[col], df_stat['Attrition'])

        # Check if contingency table is valid for chi-squared (e.g., no empty rows/columns)
        if contingency_table.min().min() == 0:
            # Handle cases where some categories might not have both Attrition outcomes
            # This can lead to issues with chi-squared. For simplicity, we'll note it.
            p_value = np.nan
            test_name = 'Chi-Squared (Invalid Table)'
        else:
            chi2, p_value, _, _ = chi2_contingency(contingency_table)
            test_name = 'Chi-Squared'

    else:
        # Numerical Feature: Mann-Whitney U test
        group_yes = df_stat[df_stat['Attrition'] == 'Yes'][col].dropna()
        group_no = df_stat[df_stat['Attrition'] == 'No'][col].dropna()

        if len(group_yes) == 0 or len(group_no) == 0:
            p_value = np.nan
            test_name = 'Mann-Whitney U (Empty Group)'
        else:
            # Mann-Whitney U test (two-sided)
            stat, p_value = mannwhitneyu(group_yes, group_no, alternative='two-sided')
            test_name = 'Mann-Whitney U'

    # Determine significance
    significant = 'N/A' if pd.isna(p_value) else ('Yes' if p_value < alpha else 'No')

    results.append({
        'Feature': col,
        'Type': df_stat[col].dtype,
        'Test': test_name,
        'P-value': f"{p_value:.4f}" if not pd.isna(p_value) else 'N/A',
        'Significant (alpha=0.05)': significant
    })

# Create and display the results DataFrame
statistical_test_results_df = pd.DataFrame(results)
display(statistical_test_results_df)


,Feature,Type,Test,P-value,Significant (alpha=0.05)
0,Age,int64,Mann-Whitney U,0.0000,Yes
1,BusinessTravel,str,Chi-Squared,0.0000,Yes
2,DailyRate,int64,Mann-Whitney U,0.0290,Yes
3,Department,str,Chi-Squared,0.0045,Yes
4,DistanceFromHome,int64,Mann-Whitney U,0.0024,Yes
5,Education,int64,Chi-Squared,0.5455,No
6,EducationField,str,Chi-Squared,0.0068,Yes
7,EnvironmentSatisfaction,int64,Chi-Squared,0.0001,Yes
8,Gender,str,Chi-Squared,0.2906,No
9,HourlyRate,int64,Mann-Whitney U,0.7976,No


# Split

In [4]:
X = df.drop(['Attrition', 'Education', 'Gender', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'], axis=1)
y = df['Attrition']

In [7]:
# Train & Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# Test & Validate
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

# Label Encoding

In [9]:
# Initialize LabelEncoder
le = LabelEncoder()

# Apply Label Encoding to categorical features in X_train, X_test, X_val
for col in X_train.select_dtypes(include='object').columns:
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    X_val[col] = le.transform(X_val[col])

# Apply Label Encoding to the target variable y_train, y_test, y_val
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
y_val = le.transform(y_val)

print("Label Encoding applied to categorical features in X and target variable y.")
print("X_train head after encoding:")
display(X_train.head())
print("y_train head after encoding:")
# display(y_train[:5])

C:\Users\Admin\AppData\Local\Temp\ipykernel_19636\811772534.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X_train.select_dtypes(include='object').columns:


Label Encoding applied to categorical features in X and target variable y.
X_train head after encoding:


,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,...,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany
1097,24,2,350,1,21,5,1,1551,3,57,...,0,14,3,2,80,3,2,3,3,1
727,18,0,287,1,5,1,1,1012,2,73,...,0,15,3,4,80,0,0,2,3,0
254,29,2,1247,2,20,2,1,349,4,45,...,0,14,3,4,80,1,10,2,3,3
1175,39,2,492,1,12,3,1,1654,4,66,...,0,21,4,3,80,0,7,3,3,5
1341,31,2,311,1,20,1,1,1881,2,89,...,0,11,3,1,80,1,10,2,3,10


y_train head after encoding:
